# Generate Patches
Chunks microscopy TIFFs into 256×256 patches with 50% overlap for annotation.

**Input:** One or more TIFF files  
**Output:** `patches.zip` containing one folder per image, each with `img_XXXX.npy` patches

```
patches/
├── image_00/
│   ├── img_0000.npy
│   ├── img_0001.npy
│   └── ...
├── image_01/
│   └── ...
```

After downloading, unzip each image folder into the repo as `imgN/patches/` and use `annotate.py` to label them.

In [ ]:
!pip install -q "opencv-python-headless>=4.9.0.80" cellpose==3.1.1.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from cellpose import io

## Config
Adjust these if your images need different patch sizes.

In [ ]:
PATCH_SIZE = 256   # patch dimensions (square)
STRIDE     = 128   # step between patches (128 = 50% overlap)

## Upload TIFFs
Upload one or more TIFF files. Each will be chunked into its own folder.

In [ ]:
from google.colab import files
uploaded = files.upload()

tiff_paths = sorted([f for f in uploaded.keys() if f.lower().endswith(('.tiff', '.tif'))])
assert len(tiff_paths) > 0, "No TIFF files uploaded"

imgs = [io.imread(f).squeeze() for f in tiff_paths]
for i, (img, path) in enumerate(zip(imgs, tiff_paths)):
    print(f"Image {i}: {os.path.basename(path)} | shape={img.shape} | dtype={img.dtype}")

In [ ]:
# Preview each image
n = len(imgs)
fig, axes = plt.subplots(1, n, figsize=(7 * n, 7))
if n == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, imgs)):
    ax.imshow(img, cmap='gray',
              vmin=np.percentile(img, 1),
              vmax=np.percentile(img, 99))
    ax.set_title(f"Image {i} — {img.shape}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## Extract Patches

In [ ]:
def extract_patches(img, patch_size=PATCH_SIZE, stride=STRIDE):
    """Chunk a 2D image into overlapping square patches."""
    H, W = img.shape[:2]
    patches = []
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patches.append(img[y:y+patch_size, x:x+patch_size])
    return patches

all_patches = []
for i, img in enumerate(imgs):
    patches = extract_patches(img)
    all_patches.append(patches)
    print(f"Image {i}: {len(patches)} patches")

print(f"\nTotal: {sum(len(p) for p in all_patches)} patches")

In [ ]:
# Preview patches from each image
for img_idx, patches in enumerate(all_patches):
    n = min(8, len(patches))
    idx = np.linspace(0, len(patches) - 1, n, dtype=int)
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 2.5))
    for col, i in enumerate(idx):
        axes[col].imshow(patches[i], cmap='gray')
        axes[col].set_title(f"{i}", fontsize=8)
        axes[col].axis('off')
    plt.suptitle(f"Image {img_idx} — sample patches", fontsize=10)
    plt.tight_layout()
    plt.show()

## Save and Download

In [ ]:
# Save each image's patches to its own folder
base_dir = '/content/patches'

for img_idx, patches in enumerate(all_patches):
    out_dir = os.path.join(base_dir, f'image_{img_idx:02d}')
    os.makedirs(out_dir, exist_ok=True)
    for i, patch in enumerate(patches):
        np.save(os.path.join(out_dir, f'img_{i:04d}.npy'), patch)
    print(f"Image {img_idx}: saved {len(patches)} patches to {out_dir}")

# Zip
!cd /content && zip -qr /content/patches.zip patches/
print(f"\nZip size: {os.path.getsize('/content/patches.zip') / 1e6:.1f} MB")

from google.colab import files
files.download('/content/patches.zip')

## Next Steps

1. Unzip and move each folder into the repo:
   ```
   patches/image_00/ → img0/patches/
   patches/image_01/ → img1/patches/
   ```

2. Generate predictions with `generate_predictions.ipynb` (optional, for model-assisted annotation)

3. Annotate:
   ```bash
   python annotate.py -i img0/patches -o img0/masks
   python annotate.py -i img0/patches -o img0/masks -p img0/pred_masks  # model-assisted
   ```

4. Fine-tune with `finetune.ipynb`